# Preprocessing

### Before running the preprocessing code, please follow the download links provided in Section 4.2 of the paper to obtain the corresponding datasets. After downloading, place the datasets in the appropriate local directories and update the file paths in this notebook accordingly.
### The preprocessed datasets are also available at https://doi.org/10.5281/zenodo.17846841.

### 1. Initializing GEE and the Urban Boundary Dataset

In [ ]:
import ee
import os
import geopandas as gpd
import pandas as pd
from datetime import datetime

ee.Authenticate()
ee.Initialize(project='your project-id')  # Replace with your GEE project ID

geojson_path = r'your_path\your_file.geojson'  # Replace with your GeoJSON file path
gdf = gpd.read_file(geojson_path)
gdf = gdf.to_crs("EPSG:4326")



In [ ]:
gdf[:10]

In [ ]:
import numpy as np
filtered_gdf = gdf[gdf['urbanArea'] > 100]
index = np.array(filtered_gdf.reset_index()['index'])
print(index)

In [ ]:
import pandas as pd
from natsort import natsorted

dir_path = r'your_path_to_climate_dataset'  # Replace with your directory path containing CSV files
target_path = r'your_path_to_final_dataset'  # Replace with your target directory path

filenames = natsorted([filename for filename in os.listdir(dir_path) if filename.endswith('.csv')])
for filename in filenames:
    if filename.endswith('.csv'):
        dataset_path = os.path.join(dir_path, filename)
        dataset = pd.read_csv(dataset_path)
        if len(dataset) == 1000:
            continue
        new_dataset = dataset[dataset['id'].isin(index)].reset_index(drop=True)
        new_dataset['id'] = [i for i in range(1020)]
        new_dataset_path = os.path.join(target_path, filename)
        new_dataset.to_csv(new_dataset_path)

        print(f"{new_dataset_path} process done.")


In [ ]:
import geopandas as gpd

geojson_path_new = r'your_path_to_filtered_urban_boundaries_geojson'  # Replace with the filtered urban-boundary GeoJSON path
gdf_new = gpd.read_file(geojson_path_new)

In [ ]:
print(gdf_new[:10])

### 2. Filtering Cities Larger Than 100 km²

In [ ]:

count = len(gdf[gdf['urbanArea'] > 100])
print(f"Number of regions with area greater than 100: {count}")
# gdf.head()

filtered_gdf = gdf[gdf['urbanArea'] > 100]

# Renumber ORIG_FID starting from 0
filtered_gdf.reset_index(drop=True, inplace=True)
filtered_gdf['ORIG_FID'] = filtered_gdf.index

geojson_path = r'your_path_to_filtered_urban_boundaries_geojson'  # Replace with the output GeoJSON path
filtered_gdf.to_file(geojson_path, driver='GeoJSON')

print(f"Filtered GeoDataFrame saved to {geojson_path}")
gdf = filtered_gdf
gdf.head()

## Data Acquisition

### 1. Creating the Geometry Dictionary and Initializing GEE

In [ ]:
import ee
import os
import geopandas as gpd
import pandas as pd
from datetime import datetime

os.environ['HTTP_PROXY'] = 'http://your_proxy_host:your_proxy_port'  # Replace with your HTTP proxy address if required
os.environ['HTTPS_PROXY'] = 'http://your_proxy_host:your_proxy_port'  # Replace with your HTTPS proxy address if required

ee.Authenticate()
ee.Initialize(project='your-project-id')  # Replace with your GEE project ID

geojson_path = r'your_path_to_filtered_urban_boundaries_geojson'  # Replace with the filtered urban-boundary GeoJSON path
gdf = gpd.read_file(geojson_path)
gdf = gdf.to_crs("EPSG:4326")

def create_gee_polygon(geometry):
    coords = list(geometry.exterior.coords) 
    polygon = ee.Geometry.Polygon(coords)
    return polygon

results = []

# geometries = [create_gee_polygon(geometry) for geometry in gdf['geometry']]
# geometry_with_ids = [{'id': i, 'geometry': geometry} for i, geometry in enumerate(geometries)]
geometry_with_ids = [{'id': i, 'geometry': create_gee_polygon(geometry)} for i, geometry in enumerate(gdf['geometry'])]

print("Geometries calulate done.")


In [ ]:
def save_geometries_and_centroids_to_csv(geometry_with_ids):
    """
    Calculate the centroid of each polygon and export the results to a local CSV file.
    The output includes `id`, `urbanArea`, and the centroid longitude and latitude.
    """
    # Extract each polygon centroid and store the data in a list
    output_dir = r"your_path_to_city_info_output_directory"  # Replace with the output directory path
    data = []
    for item in geometry_with_ids:
        geom = item['geometry']
        centroid = geom.centroid()  # Get the polygon centroid
        centroid_coords = centroid.coordinates().getInfo()  # Extract longitude and latitude
        data.append({
            'id': item['id'],
            'urbanArea': gdf.loc[item['id'], 'urbanArea'],
            'center_lon': centroid_coords[0],  # Longitude
            'center_lat': centroid_coords[1]   # Latitude
        })
    
    # Convert the records to a pandas DataFrame
    df = pd.DataFrame(data)

    # Save the results to a local CSV file
    output_file = os.path.join(output_dir, "new_city_info.csv")
    df.to_csv(output_file, index=False)

# Export the centroids
save_geometries_and_centroids_to_csv(geometry_with_ids)

### 2. Extracting UHI Data Locally

#### 2.1 Computing Shapes and Initializing the Logger

In [ ]:
from shapely.geometry import shape
import logging

raw_ids = [item["id"] for item in geometry_with_ids]
geometries = [item["geometry"] for item in geometry_with_ids]

print(len(raw_ids))
print(len(geometries))
shapes = [shape(geometry) for geometry in geometries]
print("Shapes calculate done.")

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

#### 2.2 Extracting Monthly UHI Data

In [ ]:
import ee
import os
import pandas as pd
import rasterio
import numpy as np
from rasterio.mask import mask
import concurrent.futures

output_dir = r"your_path_to_regional_uhi_output_directory"  # Replace with the output directory path
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

data_dir = r"your_path_to_uhi_raster_dataset"  # Replace with the UHI raster dataset directory

datasets = {
    "AMod2"
    # "AMod2", "Mod1", "Mod2",
    # "Myd1", "Myd2", "SAT",
    # "SMod2", "SMyd1"
}

years = range(2003, 2021, 1)
months = range(1, 13, 1)
times = ["Day"]

def save_to_csv(data, description):
    try:
        df = pd.DataFrame(data)
        output_file = os.path.join(output_dir, f"{description}.csv")
        if not os.path.exists(output_file):
            df.to_csv(output_file, index=False, mode='w', header=True)
            logger.info(f"Saved with header to {output_file}")
        else:
            df.to_csv(output_file, index=False, mode='a', header=False)
            logger.info(f"Appended data to {output_file}")
    except Exception as e:
        logger.error(f"Failed to save {description}: {e}")

def process_dataset_month(year, month, raw_ids, shapes):
    time = "Day"

    logger.info(f"Processing {year}-{month}")
    description = f"UHI_{year}_{month}"
    results = {"id": raw_ids}

    dataset_key = "Amod2"
    file_name = f"{dataset_key}/{dataset_key}_{time}_{year}_{month}.tif"
    file_path = os.path.join(data_dir, file_name)
    mean_values = []

    if not os.path.exists(file_path):
        logger.warning(f"File {file_path} doesn't exist, skipping...")
        return

    try:
        with rasterio.open(file_path) as src:
            for i, geometry in enumerate(shapes):
                out_image, out_transform = mask(src, [geometry], crop=True)
                region_data = out_image[0]
                mean_value = region_data.mean() * 0.01
                mean_values.append(mean_value)

    except Exception as e:
        logger.error(f"Process {file_path} fail: {e}")

    results.update({dataset_key: mean_values})

    save_to_csv(results, description)

def process_all_datasets():
    # TODO: Process datasets in parallel
    for year in years:
        for month in months:
            process_dataset_month(year, month, raw_ids, shapes)

process_all_datasets()


### 3. Extracting Dynamic Data

#### 3.1 Utils

In [ ]:
import math
import logging
import threading
import os
from concurrent.futures import ThreadPoolExecutor
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project='your-project-id')  # Replace with your GEE project ID

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def filter_by_month_and_calculate_mean(image_collection, variables, year, month, annual=False):
    start_date = ee.Date(f'{year}-{month:02d}-01')
    if annual == False:
        end_date = start_date.advance(1, 'month')
    else:
        end_date = ee.Date(f'{year}-12-31')
    collection = image_collection.filterDate(start_date, end_date)
    # print('Collection size:', collection.size().getInfo())
    def function(image):
        return image.set('dict', image.toDictionary(variables))
    augmented = collection.map(function)
    # processed_collection = collection.set('properties_dict', collection.toDictionary(variables))

    monthly_mean = augmented.select(variables).mean()
    return monthly_mean

def calculate_region_means(image, geometries):
    if image is None:
        return [None] * len(geometries)

    feature_collection = ee.FeatureCollection(geometries)

    def remove_geometry(feature):
        return ee.Feature(None, feature.toDictionary())
    
    # TODO: Verify that the results preserve the input order
    region_means = image.reduceRegions(
        collection=feature_collection,
        reducer=ee.Reducer.mean(),
        scale=1000
    ).map(remove_geometry)
    

    features = region_means.getInfo()['features']

    means_list = [feature['properties'] for feature in features]
    # print(means_list)
    return means_list

# Process geometries in blocks
def process_geometries_block(dataset, variables, year, month, geometries_block, ids_block, factor=1, annual=False, single=False):
    # worker_id = threading.current_thread().name
    # print(f"Worker ID: {worker_id}, Processing IDs: {ids_block[-1]}")

    if single == False:
        image = filter_by_month_and_calculate_mean(dataset, variables, year, month, annual)
    else:
        image = dataset.select(variables)
    region_means = calculate_region_means(image, geometries_block, variables)
    block_results = {original_id: {'id': original_id} for original_id in ids_block}

    for original_id, mean in zip(ids_block, region_means):
        if mean is not None:
            if len(variables) > 1:
                for i, variable in enumerate(variables):
                    block_results[original_id][variable] = mean[variable] * factor
            else:
                block_results[original_id][variables[0]] = mean['mean'] * factor

    # print(block_results)
    # [{'id': 0, ... ,'Vapour_Pressure_Mean': 14.899611792435486}, {'id': 1, ... ,'Vapour_Pressure_Mean': 17.018679100196326}]
    return list(block_results.values())


def process_all_years_and_months_parallel(years, months, dataset, variables, geometry_with_ids, output_dir, file_name, block_size=20, factor=1, annual=False, single=False):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    num_blocks = math.ceil(len(geometry_with_ids) / block_size)
    
    geometry_blocks = [
        {
            'geometries': [item['geometry'] for item in geometry_with_ids[i * block_size:(i + 1) * block_size]],
            'ids': [item['id'] for item in geometry_with_ids[i * block_size:(i + 1) * block_size]]
        }
        for i in range(num_blocks)
    ]

    for year in years:  
        for month in months:  
            if annual == False:
                logger.info(f"Processing Year: {year}, Month: {month}")
            else:
                logger.info(f"Processing Year: {year}")

            month_results = []

            with ThreadPoolExecutor() as executor:
                futures = []
                for block in geometry_blocks:
                    geometries_block = block['geometries']
                    ids_block = block['ids']
                    futures.append(executor.submit(process_geometries_block, dataset, variables, year, month, geometries_block, ids_block, factor, annual, single))

                i = 0
                for future in futures:
                    block_results = future.result()
                    month_results.extend(block_results)
                    print(f'Batch {i} done.')
                    i += 1

            # print(month_results)
            df = pd.DataFrame(month_results)
            if annual == False:
                output_file = os.path.join(output_dir, f"{file_name}_{year}_{month:02d}.csv")
            else:
                output_file = os.path.join(output_dir, f"{file_name}_{year}.csv")
            df.to_csv(output_file, index=False)
            logger.info(f"Results for {year}-{month:02d} saved to {output_file}")

#### 3.2 Extracting Meteorological Data from GEE

In [ ]:
# Each image contains daily data; calculate the mean of all images in each month
era5_daily = ee.ImageCollection('projects/climate-engine-pro/assets/ce-ag-era5/daily')

variables = [
    'Wind_Speed_10m_Mean', 'Dew_Point_Temperature_2m_Mean',
    'Relative_Humidity_2m_06h', 'Relative_Humidity_2m_15h',
    'Temperature_Air_2m_Max_24h', 'Temperature_Air_2m_Mean_24h',
    'Temperature_Air_2m_Min_24h', 'Temperature_Air_2m_Max_Day_Time',
    'Temperature_Air_2m_Mean_Day_Time', 'Temperature_Air_2m_Mean_Night_Time',
    'Temperature_Air_2m_Min_Night_Time', 'Cloud_Cover_Mean',
    'Precipitation_Rain_Duration_Fraction', 'Precipitation_Flux',
    'Snow_Thickness_Mean', 'Snow_Thickness_LWE_Mean',
    'Solar_Radiation_Flux', 'Precipitation_Solid_Duration_Fraction',
    'Vapour_Pressure_Mean'
]

output_dir = r'your_path_to_climate_data_output_directory'  # Replace with the output directory path
file_name = "region_means"
block_size = 10

years = range(2015, 2021, 1)
months = range(1, 13, 1)

process_all_years_and_months_parallel(years, months, era5_daily, variables, geometry_with_ids, output_dir, file_name, block_size)


#### 3.3 Extracting NDVI and EVI from GEE

In [ ]:
mod13a3 = ee.ImageCollection("MODIS/061/MOD13A3")

variables = ['NDVI', 'EVI']
output_dir = r'your_path_to_ndvi_evi_output_directory'  # Replace with the output directory path
file_name = "evi_ndvi"
block_size = 4

years = range(2003, 2004, 1)
months = range(1, 2, 1)

# factor 0.0001
process_all_years_and_months_parallel(years, months, mod13a3, variables, geometry_with_ids[:10], output_dir, file_name, block_size, 0.0001)

#### 3.4 Extracting PM2.5 Data from GEE

In [ ]:
pm25 = ee.ImageCollection("projects/sat-io/open-datasets/GLOBAL-SATELLITE-PM25/MONTHLY")

variables = ['b1']
output_dir = r'your_path_to_pm25_output_directory'  # Replace with the output directory path
file_name = "PM25"
block_size = 4

years = range(2013, 2014, 1)
months = range(1, 2, 1)

# annually
process_all_years_and_months_parallel(years, months, pm25, variables, geometry_with_ids[:10], output_dir, file_name, block_size)

### 4. Extracting Structural Data

#### 4.1 GHS-POP

In [ ]:
variables = ['population_count']
output_dir = r'your_path_to_ghs_pop_output_directory'  # Replace with the output directory path
file_name = "ghs_pop"
block_size = 10

for year in range(2015, 2021, 5):
    ghs_pop = ee.Image(f"JRC/GHSL/P2023A/GHS_POP/{year}")
    years = range(year, year+1, 1)
    months = range(1, 2, 1)

    # annually, single image
    process_all_years_and_months_parallel(years, months, ghs_pop, variables, geometry_with_ids, output_dir, file_name, block_size, 1, True, True)


#### 4.2 GHS-BUILT-S

In [ ]:
variables = ['built_surface']
output_dir = r'your_path_to_ghs_built_s_output_directory'  # Replace with the output directory path
file_name = "ghs_built_s"
block_size = 10

for year in range(2005, 2021, 5):
    ghs_built_s = ee.Image(f"JRC/GHSL/P2023A/GHS_BUILT_S/{year}")
    years = range(year, year+1, 1)
    months = range(1, 2, 1)

    # annually, single image
    process_all_years_and_months_parallel(years, months, ghs_built_s, variables, geometry_with_ids, output_dir, file_name, block_size, 1, True, True)


#### 4.3 GHS-BUILT-V

In [ ]:
variables = ['built_volume_total']
output_dir = r'your_path_to_ghs_built_v_output_directory'  # Replace with the output directory path
file_name = "ghs_built_v"
block_size = 10

for year in range(2005, 2021, 5):
    ghs_built_v = ee.Image(f"JRC/GHSL/P2023A/GHS_BUILT_V/{year}")
    years = range(year, year+1, 1)
    months = range(1, 2, 1)

    # annually, single image
    process_all_years_and_months_parallel(years, months, ghs_built_v, variables, geometry_with_ids, output_dir, file_name, block_size, 1, True, True)

#### 4.4 GHS-BUILT-H

In [ ]:
variables = ['built_height']
output_dir = r'your_path_to_ghs_built_h_output_directory'  # Replace with the output directory path
file_name = "ghs_built_h"
block_size = 10

ghs_built_h = ee.Image(f"JRC/GHSL/P2023A/GHS_BUILT_H/2018")
years = range(2018, 2019, 1)
months = range(1, 2, 1)

# annually, single image
process_all_years_and_months_parallel(years, months, ghs_built_h, variables, geometry_with_ids, output_dir, file_name, block_size, 1, True, True)

#### 4.5 GHS-BUILT-C

In [ ]:
category_mapping = {
    "0": "No data",
    "1": "open spaces, low vegetation surfaces",
    "2": "open spaces, medium vegetation surfaces",
    "3": "open spaces, high vegetation surfaces",
    "4": "open spaces, water surfaces",
    "5": "open spaces, road surfaces",
    "11": "built spaces, residential, building height <= 3m",
    "12": "built spaces, residential, 3m < building height <= 6m",
    "13": "built spaces, residential, 6m < building height <= 15m",
    "14": "built spaces, residential, 15m < building height <= 30m",
    "15": "built spaces, residential, building height > 30m",
    "21": "built spaces, non-residential, building height <= 3m",
    "22": "built spaces, non-residential, 3m < building height <= 6m",
    "23": "built spaces, non-residential, 6m < building height <= 15m",
    "24": "built spaces, non-residential, 15m < building height <= 30m",
    "25": "built spaces, non-residential, building height > 30m",
}
# print(category_mapping.keys())

def calculate_region_means(image, geometries, variables):
    if image is None:
        return [None] * len(geometries)

    feature_collection = ee.FeatureCollection(geometries)

    def remove_geometry(feature):
        return ee.Feature(None, feature.toDictionary())
    
    # TODO: Verify that the results preserve the input order
    region_means = image.reduceRegions(
        collection=feature_collection,
        reducer=ee.Reducer.frequencyHistogram(),
        scale=10
    ).map(remove_geometry)

    features = region_means.aggregate_array('histogram').getInfo()

    means_list = []
    
    for index, row in enumerate(features):
        temp = {key: [] for key in category_mapping.keys()}
        total_pixels = sum(row.values())
        for key in category_mapping.keys():
            if key in row:
                temp[key].append((row[key] / total_pixels) * 100 if total_pixels > 0 else 0)
            else:
                temp[key].append(0)
                
        means_list.append(temp)
    
    # print(means_list)
    return means_list

def process_geometries_block(dataset, variables, year, month, geometries_block, ids_block, factor=1, annual=False, single=False):
    # worker_id = threading.current_thread().name
    # print(f"Worker ID: {worker_id}, Processing IDs: {ids_block[-1]}")

    if single == False:
        image = filter_by_month_and_calculate_mean(dataset, variables, year, month, annual)
    else:
        image = dataset.select(variables)
    region_means = calculate_region_means(image, geometries_block, variables)
    block_results = {original_id: {'id': original_id} for original_id in ids_block}

    for original_id, mean in zip(ids_block, region_means):
        if mean is not None:
            for i, variable in enumerate(category_mapping.keys()):
                block_results[original_id][variable] = mean[variable][0] * factor
            # if len(variables) > 1:
            #     for i, variable in enumerate(variables):
            #         block_results[original_id][variable] = mean[variable] * factor
            # else:
            #     block_results[original_id][variables[0]] = mean['mean'] * factor

    # print(block_results)
    # [{'id': 0, ... ,'Vapour_Pressure_Mean': 14.899611792435486}, {'id': 1, ... ,'Vapour_Pressure_Mean': 17.018679100196326}]
    return list(block_results.values())


variables = ['built_characteristics']
output_dir = r'your_path_to_ghs_built_c_output_directory'  # Replace with the output directory path
file_name = "ghs_built_c"
block_size = 4

ghs_built_c = ee.Image(f"JRC/GHSL/P2023A/GHS_BUILT_C/2018")
years = range(2018, 2019, 1)
months = range(1, 2, 1)

# annually, single image
process_all_years_and_months_parallel(years, months, ghs_built_c, variables, geometry_with_ids, output_dir, file_name, block_size, 1, True, True)

#### 4.6 GHS-SMOD

In [ ]:
variables = ['smod_code']
output_dir = r'your_path_to_ghs_smod_output_directory'  # Replace with the output directory path
file_name = "ghs_smod"
block_size = 10

for year in range(2005, 2021, 5):
    ghs_built_smod = ee.Image(f"JRC/GHSL/P2023A/GHS_SMOD_V2-0/{year}")
    years = range(year, year+1, 1)
    months = range(1, 2, 1)

    # annually, single image
    process_all_years_and_months_parallel(years, months, ghs_built_smod, variables, geometry_with_ids, output_dir, file_name, block_size, 1, True, True)

#### 4.7 Urban Morphology Metrics

In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon

# Calculate the fractal dimension (Df) using the box-counting method
def box_counting_fractal_dimension(geometry, scales):
    if geometry.is_empty or geometry.is_valid is False:
        return np.nan

    box_counts = []
    for scale in scales:
        min_x, min_y, max_x, max_y = geometry.bounds
        x_coords = np.arange(min_x, max_x, scale)
        y_coords = np.arange(min_y, max_y, scale)
        count = 0
        for x in x_coords:
            for y in y_coords:
                box = Polygon([
                    (x, y), (x + scale, y), 
                    (x + scale, y + scale), (x, y + scale)
                ])
                if geometry.intersects(box):
                    count += 1
        box_counts.append(count)

    box_counts = np.array(box_counts)
    valid = box_counts > 0
    if valid.sum() < 2:
        return np.nan
    log_scales = -np.log(scales[valid])
    log_counts = np.log(box_counts[valid])
    slope, _ = np.polyfit(log_scales, log_counts, 1)
    return slope

# Log area, fractal dimension, anisometry, shape index, compactness, and edge density
def calculate_city_metrics(gdf):

    results = []

    scales = np.logspace(-3, 0, 10)
    
    gdf_utm = gdf.to_crs("EPSG:32633")
    

    for idx, row in gdf.iterrows():
        geometry = row['geometry']
        area = row['urbanArea'] 
        
        # gdf_utm = geometry.to_crs("EPSG:32633")

        perimeter = gdf_utm.iloc[idx]['geometry'].length / 1000

        log_area = np.log(area)

        # Fractal dimension
        # fractal_dimension = box_counting_fractal_dimension(geometry, scales)

        bounds = geometry.minimum_rotated_rectangle.bounds
        major_axis = max(bounds[2] - bounds[0], bounds[3] - bounds[1])
        minor_axis = min(bounds[2] - bounds[0], bounds[3] - bounds[1])
        anisometry = major_axis / minor_axis if minor_axis != 0 else np.nan

        # Mean shape index (MSI)
        msi = perimeter / np.sqrt(area)

        # Compactness index (CI)
        print(area)
        print(perimeter)
        
        a_circle = (perimeter ** 2) / (4 * np.pi)
        compactness = area / a_circle if a_circle != 0 else np.nan

        # Edge density (ED)
        edge_density = perimeter / area

        centroid = geometry.centroid

        result = {
            'id': idx,
            'centroid_x': centroid.x,
            'centroid_y': centroid.y,
            'Area': row['urbanArea'],
            'Log Area': log_area,
            # 'Fractal Dimension': fractal_dimension,
            'Anisometry': np.log(anisometry) if anisometry > 0 else np.nan,
            'Shape Index (MSI)': msi,
            'Compactness (CI)': compactness,
            'Edge Density (ED)': edge_density
        }
        print(result)
        results.append(result)

    return gpd.GeoDataFrame(results)

metrics = calculate_city_metrics(gdf[:10])

# output_dir = r'your_path_to_city_metrics_output_directory'  # Replace with the output directory path
# if not os.path.exists(output_dir):
#     os.makedirs(output_dir)
# output_path = os.path.join(output_dir, 'city_metrics.csv')

# metrics.to_csv(output_path, index=False)
# print(f"Results saved to {output_path}")


In [ ]:
import pandas as pd
import os

def collect_city_data(input_dir, city_id, output_dir):

    all_city_data = []
    for year in range(2003, 2021):
        for month in range(1, 13):
            file_name = f"UHI_{year}_{month}.csv"
            file_path = os.path.join(input_dir, file_name)

            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                city_data = df[df['id'] == city_id]
                all_city_data.append(city_data)
            else:
                print(f"File {file_name} does not exist!")

    if all_city_data:
        all_city_data_df = pd.concat(all_city_data, ignore_index=True)
        output_file = os.path.join(output_dir, f'UHI_city_{city_id}.csv')
        all_city_data_df.to_csv(output_file, index=False)
        print(f"Saved data for city ID {city_id} to {output_file}")
    else:
        print(f"No data found for city ID {city_id}.")

def process_all_city_ids(input_dir, output_dir):

    sample_file = os.path.join(input_dir, "UHI_2003_1.csv")
    if os.path.exists(sample_file):
        df = pd.read_csv(sample_file)
        city_ids = df['id'].unique()
    else:
        print(f"File {sample_file} does not exist!")
        return

    for city_id in city_ids:
        print(f"Processing data for city ID {city_id}...")
        collect_city_data(input_dir, city_id, output_dir)

if __name__ == "__main__":
    input_directory = r"your_path_to_regional_uhi_input_directory"  # Replace with the input directory path

    output_directory = r"your_path_to_extracted_city_data_output_directory"  # Replace with the output directory path
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    process_all_city_ids(input_directory, output_directory)


In [ ]:
import pandas as pd

df = pd.read_csv('your_path_to_city_metrics_csv')  # Replace with the city metrics CSV path
df

In [ ]:
df_sampled = df.sample(n=500, random_state=123, axis=0)
df_sampled.to_csv('your_path_to_sampled_city_metrics_csv', index=False)  # Replace with the output CSV path
df_sampled

In [ ]:
df_test_set = df.drop(index=df_sampled.index)
df_test_set.to_csv('your_path_to_city_metrics_test_set_csv')  # Replace with the output CSV path

In [ ]:
df_test = df_test_set.sample(n=100, random_state=256, axis=0)
df_test.to_csv('your_path_to_city_metrics_test_csv', index=False)  # Replace with the output CSV path
df_test

In [ ]:
df_test['id'].tolist()

In [ ]:
import pandas as pd

df_train = pd.read_csv(r"your_path_to_city_metrics_training_csv")  # Replace with the training CSV path
df_train_100city = df_train.sample(n=100, random_state=128, axis=0)
df_train_100city.to_csv(r"your_path_to_100_city_training_csv", index=False)  # Replace with the output CSV path
df_train_100city_1 = df_train_100city.sample(n=80, random_state=128, axis=0)
df_train_100city_2 = df_train_100city.sample(n=80, random_state=256, axis=0)
df_train_100city_1.to_csv(r"your_path_to_100_city_training_subset_1_csv", index=False)  # Replace with the output CSV path
df_train_100city_2.to_csv(r"your_path_to_100_city_training_subset_2_csv", index=False)  # Replace with the output CSV path